# RCM Denial Prediction — Demo Sandbox

**Project:** `ai-project-q2w5uxlkh4c6o` (Foundry / Azure ML)
**Model:** `rcm-denial-prediction-space:1` (mlflow_model)
**Datasets:** `claims_training:1`, `denials_gold:1` — synthetic, no PHI.

This notebook is a presenter sandbox: load the registered model, score a synthetic claim, and inspect the denial probability + recommended code. Replace the inline `CLAIM_TEXT` to try other examples.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

SUBSCRIPTION_ID = "a7fecb91-4553-4aca-976e-274add998c8d"
RESOURCE_GROUP = "rg-rag-project-dev"
WORKSPACE = "ai-project-q2w5uxlkh4c6o"

ml = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print("workspace:", ml.workspace_name)
print("claims_training:", ml.data.get(name="claims_training", version="1").path)
print("denials_gold:", ml.data.get(name="denials_gold", version="1").path)
print("model:", ml.models.get(name="rcm-denial-prediction-space", version="1").id)

In [ ]:
import os, mlflow

model_uri = f"models:/rcm-denial-prediction-space/1"
local_path = mlflow.artifacts.download_artifacts(artifact_uri=model_uri)
print("model artifacts:", os.listdir(local_path))
with open(os.path.join(local_path, "MLmodel"), "r") as fh:
    print(fh.read())

In [ ]:
# Demo-only: this stub model is a placeholder for the real PubMedBERT denial predictor.
# In the live demo we'll wire to a real scoring endpoint. For now, illustrate the I/O contract.
import json, hashlib

CLAIM_TEXT = "Patient with chronic kidney disease, missing prior auth for outpatient dialysis."

def score_stub(text: str) -> dict:
    h = int(hashlib.sha256(text.encode()).hexdigest(), 16)
    prob = round(0.45 + (h % 4500) / 10000.0, 4)
    code = ["CO-50", "CO-197", "PR-1", "CO-16"][h % 4]
    return {"claim_text": text, "denial_probability": prob, "denial_code": code}

result = score_stub(CLAIM_TEXT)
print(json.dumps(result, indent=2))

## Next steps

- Replace `score_stub` with `mlflow.pyfunc.load_model(model_uri).predict(...)` once the real PubMedBERT weights ship.
- Mount `denials_gold` as an `mltable` for a small batch eval.
- Promote the winning version to `rcm-denial-prediction-space:2` once gated by governance.